# Power-Law Models

Fit Power-Law curves to slurry viscosity measurements and build a row-level physics dataset for downstream modelling.

## 1. Import Libraries

In [1]:
import warnings
import numpy as np
import pandas as pd

from scipy.optimize import least_squares
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C, WhiteKernel
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

## 2. Load Data

In [2]:
df = pd.read_csv('../data/processed/combined_slurry_data_expanded.csv')

print(f"Rows: {len(df)}  |  Unique formulations (Composite_Mix_ID): {df['Composite_Mix_ID'].nunique()}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Rows: 178  |  Unique formulations (Composite_Mix_ID): 68
Columns: ['Dispersent_Type', 'Solid_Content_pct', 'Solid_Additive_pct', 'Viscosity_at_shear_rate_1_1/s', 'Viscosity_at_shear_rate_10_1/s', 'Viscosity_at_shear_rate_100_1/s', 'Source_Batch', 'Composite_Mix_ID', 'NMC_pct', 'C65_pct', 'KS6L_pct', 'PVDF_pct']


,Dispersent_Type,Solid_Content_pct,Solid_Additive_pct,Viscosity_at_shear_rate_1_1/s,Viscosity_at_shear_rate_10_1/s,Viscosity_at_shear_rate_100_1/s,Source_Batch,Composite_Mix_ID,NMC_pct,C65_pct,KS6L_pct,PVDF_pct
0,Hypermer,73.0,0.50,10.56640,3.78921,1.99755,Batch_1,F1_Hypermer_73.0_0.5,96.0,2.0,0.0,2.0
1,Hypermer,73.0,0.25,71.65190,14.08460,4.82515,Batch_1,F1_Hypermer_73.0_0.25,96.0,2.0,0.0,2.0
2,Hypermer,73.0,0.25,9.64639,3.26827,1.61720,Batch_1,F1_Hypermer_73.0_0.25,96.0,2.0,0.0,2.0
3,Hypermer,77.0,0.25,61.11070,18.77450,7.51220,Batch_1,F1_Hypermer_77.0_0.25,96.0,2.0,0.0,2.0
4,Hypermer,73.0,0.25,8.37111,4.83186,2.30422,Batch_1,F1_Hypermer_73.0_0.25,96.0,2.0,0.0,2.0


## 3. Define Power-Law Model

In [3]:
def power_law_viscosity(shear_rate, k_consistency, n_flow):
    """
    Power-Law model for apparent viscosity: eta = K * gamma_dot^(n - 1).

    Parameters
    ----------
    shear_rate    : array-like  -- applied shear rate (1/s)
    k_consistency : float       -- consistency index K
    n_flow        : float       -- flow index n (<1 shear-thinning)
    """
    return k_consistency * np.power(shear_rate, n_flow - 1.0)

## 4. Set Up Columns and Features

In [4]:
shear_rates = np.array([1.0, 10.0, 100.0], dtype=float)

target_cols = [
    'Viscosity_at_shear_rate_1_1/s',
    'Viscosity_at_shear_rate_10_1/s',
    'Viscosity_at_shear_rate_100_1/s',
]

eps = 1e-8
df['Solid_to_Liquid_Ratio'] = df['Solid_Content_pct'] / (100.0 - df['Solid_Content_pct'] + eps)
df['Binder_to_Active_Ratio'] = df['PVDF_pct'] / (df['NMC_pct'] + df['C65_pct'] + eps)

input_cols = [
    'Dispersent_Type', 'Solid_Content_pct', 'Solid_Additive_pct',
    'NMC_pct', 'C65_pct', 'KS6L_pct', 'PVDF_pct', 'Source_Batch',
    'Solid_to_Liquid_Ratio', 'Binder_to_Active_Ratio',
]

print('Target columns:', target_cols)
print('Input columns :', input_cols)
display(df[input_cols + ['Composite_Mix_ID']].head())

Target columns: ['Viscosity_at_shear_rate_1_1/s', 'Viscosity_at_shear_rate_10_1/s', 'Viscosity_at_shear_rate_100_1/s']
Input columns : ['Dispersent_Type', 'Solid_Content_pct', 'Solid_Additive_pct', 'NMC_pct', 'C65_pct', 'KS6L_pct', 'PVDF_pct', 'Source_Batch', 'Solid_to_Liquid_Ratio', 'Binder_to_Active_Ratio']


,Dispersent_Type,Solid_Content_pct,Solid_Additive_pct,NMC_pct,C65_pct,KS6L_pct,PVDF_pct,Source_Batch,Solid_to_Liquid_Ratio,Binder_to_Active_Ratio,Composite_Mix_ID
0,Hypermer,73.0,0.50,96.0,2.0,0.0,2.0,Batch_1,2.703704,0.020408,F1_Hypermer_73.0_0.5
1,Hypermer,73.0,0.25,96.0,2.0,0.0,2.0,Batch_1,2.703704,0.020408,F1_Hypermer_73.0_0.25
2,Hypermer,73.0,0.25,96.0,2.0,0.0,2.0,Batch_1,2.703704,0.020408,F1_Hypermer_73.0_0.25
3,Hypermer,77.0,0.25,96.0,2.0,0.0,2.0,Batch_1,3.347826,0.020408,F1_Hypermer_77.0_0.25
4,Hypermer,73.0,0.25,96.0,2.0,0.0,2.0,Batch_1,2.703704,0.020408,F1_Hypermer_73.0_0.25


## 5. Fit Power-Law Parameters Per Row

In [11]:
USE_LOG_RESIDUALS_FOR_PL_FIT = True
RESIDUAL_EPS = 1e-8

pl_dataset = []
failed_fits = []

def fit_row_power_law(x_values, y_values, initial_guess, bounds, reg_weight=1e-2):
    def residuals(theta):
        k_consistency = theta[0]
        n_flow = theta[1]

        y_hat = power_law_viscosity(x_values, k_consistency, n_flow)

        if USE_LOG_RESIDUALS_FOR_PL_FIT:
            data_resid = np.log(y_values + RESIDUAL_EPS) - np.log(y_hat + RESIDUAL_EPS)
        else:
            data_resid = y_values - y_hat

        reg_resid = np.sqrt(reg_weight) * (theta - initial_guess)
        return np.concatenate([data_resid, reg_resid])

    return least_squares(residuals, x0=initial_guess, bounds=bounds, max_nfev=20000)

for row_idx, raw_row in df.iterrows():
    x_data = shear_rates
    y_data = raw_row[target_cols].values.astype(float)

    try:
        initial_guess = np.array([np.max(y_data), 0.5], dtype=float)
        bounds = (
            np.array([1e-4, 0.05], dtype=float),
            np.array([1e4, 1.5], dtype=float),
        )

        final_fit = fit_row_power_law(x_data, y_data, initial_guess, bounds)

        k_consistency = final_fit.x[0]
        n_flow = final_fit.x[1]

        y_pred_all = power_law_viscosity(x_data, k_consistency, n_flow)
        pl_rmse = np.sqrt(mean_squared_error(y_data, y_pred_all))
        pl_r2 = r2_score(y_data, y_pred_all)
        pl_mae = mean_absolute_error(y_data, y_pred_all)

        record = raw_row[input_cols].to_dict()
        record['row_index'] = row_idx
        record['Composite_Mix_ID'] = raw_row['Composite_Mix_ID']
        record['k_consistency'] = k_consistency
        record['n_flow'] = n_flow
        record['PL_N_RMSE'] = pl_rmse
        record['PL_N_R2'] = pl_r2
        record['PL_N_MAE'] = pl_mae

        pl_dataset.append(record)
    except Exception:
        failed_fits.append(row_idx)

print(f'Successful fits : {len(pl_dataset)}')
print(f'Failed fits     : {len(failed_fits)}')

Successful fits : 178
Failed fits     : 0


## 6. Build Physics Dataset

In [12]:
physics_df = pd.DataFrame(pl_dataset)
physics_df['pl_confidence'] = 1.0 / (physics_df['PL_N_RMSE'] + 1e-6)

print(f'Physics dataset shape: {physics_df.shape}')
print(f"Unique formulations : {physics_df['Composite_Mix_ID'].nunique()}")
print('\nPower-Law parameter and fit summary:')
display(physics_df[['k_consistency', 'n_flow', 'PL_N_RMSE', 'PL_N_R2', 'PL_N_MAE']].describe().round(4))

print('\nDataset preview:')
display(physics_df.head())

Physics dataset shape: (178, 18)
Unique formulations : 68

Power-Law parameter and fit summary:


,k_consistency,n_flow,PL_N_RMSE,PL_N_R2,PL_N_MAE
count,178.0000,178.0000,178.0000,178.0000,178.0000
mean,52.7751,0.5769,1.1374,0.9911,0.7858
std,66.1894,0.1874,1.4024,0.0131,0.8983
min,1.1330,0.0655,0.0024,0.9026,0.0023
25%,5.8986,0.4240,0.1547,0.9897,0.1404
50%,23.5947,0.5926,0.5646,0.9953,0.4457
75%,72.3869,0.7245,1.5736,0.9981,1.0801
max,324.6596,0.9191,6.6738,1.0000,4.2679



Dataset preview:


,Dispersent_Type,Solid_Content_pct,Solid_Additive_pct,NMC_pct,C65_pct,KS6L_pct,PVDF_pct,Source_Batch,Solid_to_Liquid_Ratio,Binder_to_Active_Ratio,row_index,Composite_Mix_ID,k_consistency,n_flow,PL_N_RMSE,PL_N_R2,PL_N_MAE,pl_confidence
0,Hypermer,73.0,0.50,96.0,2.0,0.0,2.0,Batch_1,2.703704,0.020408,0,F1_Hypermer_73.0_0.5,10.213910,0.630344,0.395437,0.988518,0.353252,2.528839
1,Hypermer,73.0,0.25,96.0,2.0,0.0,2.0,Batch_1,2.703704,0.020408,1,F1_Hypermer_73.0_0.25,71.499686,0.390612,2.038518,0.995245,1.382722,0.490552
2,Hypermer,73.0,0.25,96.0,2.0,0.0,2.0,Batch_1,2.703704,0.020408,2,F1_Hypermer_73.0_0.25,9.302837,0.605160,0.346166,0.990002,0.310140,2.888777
3,Hypermer,77.0,0.25,96.0,2.0,0.0,2.0,Batch_1,3.347826,0.020408,3,F1_Hypermer_77.0_0.25,61.026951,0.533695,1.222712,0.997192,0.849892,0.817853
4,Hypermer,73.0,0.25,96.0,2.0,0.0,2.0,Batch_1,2.703704,0.020408,4,F1_Hypermer_73.0_0.25,8.541647,0.722823,0.214229,0.992587,0.189858,4.667878


## 7. Train-Test Split

In [13]:
unique_formulations = physics_df['Composite_Mix_ID'].dropna().unique()
form_train, form_test = train_test_split(
    unique_formulations,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

physics_train = physics_df[physics_df['Composite_Mix_ID'].isin(form_train)].copy()
physics_test = physics_df[physics_df['Composite_Mix_ID'].isin(form_test)].copy()

train_forms = set(physics_train['Composite_Mix_ID'].unique())
test_forms = set(physics_test['Composite_Mix_ID'].unique())
overlap_forms = train_forms & test_forms

print(f'Train set: {len(physics_train)} rows  |  {len(train_forms)} formulations')
print(f'Test set : {len(physics_test)} rows  |  {len(test_forms)} formulations')
print(f'Formulation overlap (must be 0): {len(overlap_forms)}')

Train set: 136 rows  |  54 formulations
Test set : 42 rows  |  14 formulations
Formulation overlap (must be 0): 0


## 8. ML Model Training

In [14]:
ml_feature_cols = input_cols
ml_target_cols = ['k_consistency', 'n_flow']

X_train = physics_train[ml_feature_cols].copy()
X_test = physics_test[ml_feature_cols].copy()
y_train_raw = physics_train[ml_target_cols].copy()
y_test_raw = physics_test[ml_target_cols].copy()

TARGET_LOG_EPS = 1e-8
y_train_model = np.log(y_train_raw.clip(lower=TARGET_LOG_EPS))

sample_weight_train = physics_train['pl_confidence'].copy()
sample_weight_train = sample_weight_train / sample_weight_train.mean()
sample_weight_train = sample_weight_train.clip(lower=0.25, upper=4.0)
sample_noise_train = 1e-4 / sample_weight_train.values

X_train_encoded = pd.get_dummies(X_train, columns=['Dispersent_Type', 'Source_Batch'], drop_first=False)
X_test_encoded = pd.get_dummies(X_test, columns=['Dispersent_Type', 'Source_Batch'], drop_first=False)

missing_cols = set(X_train_encoded.columns) - set(X_test_encoded.columns)
for col in missing_cols:
    X_test_encoded[col] = 0
X_test_encoded = X_test_encoded[X_train_encoded.columns]

X_train_encoded = X_train_encoded.fillna(X_train_encoded.mean())
X_test_encoded = X_test_encoded.fillna(X_train_encoded.mean())

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

print(f'ML training set: {X_train.shape}')
print(f'ML test set:     {X_test.shape}')
print(f'Targets: {ml_target_cols}')

ML training set: (136, 10)
ML test set:     (42, 10)
Targets: ['k_consistency', 'n_flow']


### 8.1 Gaussian Process Regression

In [15]:
gpr_models = {}
gpr_train_r2 = {}
gpr_test_r2 = {}

for target_param in ml_target_cols:
    kernel = C(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, length_scale_bounds=(1e-2, 1e2), nu=2.5) + WhiteKernel(noise_level=0.05, noise_level_bounds=(1e-6, 1e1))
    gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, alpha=sample_noise_train, random_state=42)

    gpr.fit(X_train_scaled, y_train_model[target_param])
    gpr_models[target_param] = gpr

    y_pred_train = np.exp(gpr.predict(X_train_scaled))
    y_pred_test = np.exp(gpr.predict(X_test_scaled))

    gpr_train_r2[target_param] = r2_score(y_train_raw[target_param], y_pred_train)
    gpr_test_r2[target_param] = r2_score(y_test_raw[target_param], y_pred_test)

print('GPR Model Performance:')
print('\nTrain R2 scores:')
for param, r2 in gpr_train_r2.items():
    print(f'  {param:15s}: {r2:.4f}')

print('\nTest R2 scores:')
for param, r2 in gpr_test_r2.items():
    print(f'  {param:15s}: {r2:.4f}')

GPR Model Performance:

Train R2 scores:
  k_consistency  : 0.7483
  n_flow         : 0.6057

Test R2 scores:
  k_consistency  : 0.6849
  n_flow         : 0.4718


### 8.2 Random Forest and Extra Trees

In [16]:
rf_train_r2, rf_test_r2 = {}, {}
et_train_r2, et_test_r2 = {}, {}

for target_param in ml_target_cols:
    rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
    rf.fit(X_train_encoded, y_train_model[target_param], sample_weight=sample_weight_train.values)
    rf_train_r2[target_param] = r2_score(y_train_raw[target_param], np.exp(rf.predict(X_train_encoded)))
    rf_test_r2[target_param] = r2_score(y_test_raw[target_param], np.exp(rf.predict(X_test_encoded)))

    et = ExtraTreesRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
    et.fit(X_train_encoded, y_train_model[target_param], sample_weight=sample_weight_train.values)
    et_train_r2[target_param] = r2_score(y_train_raw[target_param], np.exp(et.predict(X_train_encoded)))
    et_test_r2[target_param] = r2_score(y_test_raw[target_param], np.exp(et.predict(X_test_encoded)))

comparison = pd.DataFrame({
    'Parameter': ml_target_cols,
    'GPR_Train_R2': [gpr_train_r2[p] for p in ml_target_cols],
    'GPR_Test_R2': [gpr_test_r2[p] for p in ml_target_cols],
    'RF_Train_R2': [rf_train_r2[p] for p in ml_target_cols],
    'RF_Test_R2': [rf_test_r2[p] for p in ml_target_cols],
    'ET_Train_R2': [et_train_r2[p] for p in ml_target_cols],
    'ET_Test_R2': [et_test_r2[p] for p in ml_target_cols],
})

display(comparison.round(4))

,Parameter,GPR_Train_R2,GPR_Test_R2,RF_Train_R2,RF_Test_R2,ET_Train_R2,ET_Test_R2
0,k_consistency,0.7483,0.6849,0.7933,0.6526,0.8055,0.6047
1,n_flow,0.6057,0.4718,0.7630,0.3698,0.7741,0.4105
